# Function Calling


https://platform.openai.com/docs/guides/function-calling

<img src="https://cdn.openai.com/API/docs/images/function-calling-diagram-steps.png" alt="Function Calling Diagram" width="600"/>

Function Calling은 OpenAI의 GPT 모델이 특정 작업을 수행할 수 있도록 함수 호출을 지원하는 기능이다.

모델이 함수 이름과 JSON 인자를 선택해 애플리케이션에 전달하고, 애플리케이션은 호출을 검사하고 로컬 함수나 외부 API를 실행한다.

그리고 실행 결과를 모델에 돌려보내 최종 자연어 응답을 만든다.

이 기능을 활용하면 GPT 모델이 단순한 텍스트 생성뿐만 아니라 더 복잡한 작업도 자동으로 처리할 수 있다.

모델이 만든 인자는 신뢰하지 않는다. 허용 함수, 인자, 권한을 애플리케이션에서 검증한다.


## Function Calling 처리 순서

Function Calling은 모델과 애플리케이션이 도구 실행 정보를 주고받는 약속이다.

처리 단계는 다음과 같다.

1. 애플리케이션이 함수 이름, 설명, JSON Schema를 모델에 보낸다.
2. 모델이 선택한 함수 이름과 JSON 인자를 반환한다.
3. 애플리케이션이 허용된 함수만 실행한다.
4. 애플리케이션이 함수 결과를 같은 호출 ID와 함께 모델에 보낸다.
5. 모델이 도구 결과를 근거로 최종 텍스트를 생성한다.

`strict=True`는 정의한 JSON Schema에 맞는 인자를 받기 위한 권장 설정이다.

- 모든 속성을 `required`에 넣는다.
- `additionalProperties`를 `false`로 설정한다.
- 반환된 이름과 인자는 실행 전에 다시 검사한다.

공식 문서는 다음과 같다.

- [Function Calling 가이드](https://developers.openai.com/api/docs/guides/function-calling)
- [Responses 생성 API](https://developers.openai.com/api/reference/resources/responses/methods/create)
- [Chat Completions 생성 API](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)


### API 키가 있는 로컬 실행 환경 준비

이미 학습한 `.env` 설정으로 `OPENAI_API_KEY`와 `OPENWEATHER_API_KEY`를 현재 커널에 등록한다. 키 값은 출력하지 않으며, 아래 함수와 OpenAI 클라이언트가 환경 변수에서 필요한 값을 읽는다.


In [3]:
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
import os

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("08_llm 프로젝트 최상위에 .env 파일을 만든 뒤 다시 실행한다.")

load_dotenv(dotenv_path, override=False)
client = OpenAI()

OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

if not os.getenv("OPENWEATHER_API_KEY") or not OPENWEATHER_API_KEY:
    raise RuntimeError('.env에 API KEY 누갈됨')

print("API KEY 불러오기 완료")

API KEY 불러오기 완료


## 날씨 함수와 입력 스키마

아래 함수는 도시와 단위 입력을 OpenWeatherMap API 요청으로 변환하고, 실제 응답에서 필요한 필드만 JSON 문자열로 반환한다. 이후 도구 호출 루프는 이 문자열을 모델에 전달하므로, API 응답 전체가 아니라 위치·날씨·온도·습도만 선택한다.


### 실제 날씨 응답을 도구 결과 JSON으로 정규화하기

함수 입력 `city`, `units`는 OpenWeatherMap의 URL 쿼리로 전달된다. HTTP 응답에서 필요한 날씨 필드만 골라 JSON 문자열로 만들며, 이 문자열은 뒤의 `tool` 메시지 또는 `function_call_output`의 결과가 된다.


In [6]:
import requests
import json
def get_current_weather(city, units):

    # 현재 날씨 한 건을 조회하는 OpenWeatherMap의 엔드포인트
    url = "http://api.openweathermap.org/data/2.5/weather"

    # 파라미터 작성
    params = {
        'q' : city,                         # 도시명
        'appid' : OPENWEATHER_API_KEY,      # API KEY
        'units' : units                     # 온도 단위 (C, F)
    }

    # 요청 후 응답 받기 (Response 객체)
    # timeout = 10 -> 10초 내에 응답이 없다면 오류
    response = requests.get(url, params = params, timeout = 10)

    # 응답 데이터(JSON 양식의 텍스트) -> dict/list로 역직렬화
    data = response.json()

    # 정상 응답(HTTP 응답 상태 코드 200)인 경우
    if response.status_code == 200:
        weather_info = {'location' : city,
                        # 현재 날씨 설명
                        'weather' : data['weather'][0]['description'],
                        # 현재 온도
                        'temperature' : data['main']['temp'],
                        # 현재 습도
                        'humidity' : data['main']['humidity'],
                        # 온도 단위 (celsius : 섭씨, fahrenheit : 화씨)
                        'units' : 'celsius' if units == 'metric' else 'fahrenheit'
                        }
    # 응답 실패
    else:
        weather_info = {'location' : city,
                        'error' : 'weather lookup failed'}

    # ensure_ascii = False -> 한글을 \u000 형태로 바꾸지 않고 그대로 유지
    return json.dumps(weather_info, ensure_ascii = False)

# 호출 테스트
get_current_weather("Seoul", "metric")

'{"location": "Seoul", "weather": "few clouds", "temperature": 35.93, "humidity": 49, "units": "celsius"}'

## Chat Completions 도구 호출 루프

Function Calling에서 모델은 외부 API나 Python 함수를 직접 실행하지 않는다. 사용자의 질문을 보고 **어떤 함수가 필요한지, 그 함수에 어떤 인자를 넘겨야 하는지**를 구조화된 형태로 알려준다.

1. 사용자가 `오늘 서울 날씨를 알려 줘.`라고 질문하면
2. 모델은 날씨 조회에 필요한 함수와 인자를 선택한다.
3. Python 코드는 선택된 함수와 인자를 사용해 OpenWeatherMap API를 호출하고, 조회 결과를 모델에게 돌려준다.
4. 모델은 전달받은 날씨 정보를 자연어로 정리해 사용자에게 답변한다.

여기서 **도구 호출 루프**는 `모델에게 함수 설명 전달 → 모델이 호출 요청 반환 → Python이 함수 실행 → 결과를 모델에 재전달 → 최종 답변 생성`으로 이어지는 왕복 과정을 뜻한다.

### 코드에서 사용하는 값

- `messages`: 사용자 질문, 모델의 함수 호출 요청, 함수 실행 결과를 순서대로 저장하는 대화 이력이다.
- `tools`: 모델이 사용할 수 있는 함수의 **사용 설명서**이다. 함수 이름, 기능, 받을 인자의 형식을 첫 번째 요청과 함께 전달한다.
- `tool_calls`: `tools`를 확인한 모델이 반환하는 **함수 호출 요청**이다. 호출할 함수 이름, JSON 인자, 호출 ID가 들어 있다.
- `available_functions`: 모델이 반환한 함수 이름을 실제 Python 함수와 연결하는 허용 목록이다.
- `role="tool"`: Python 함수의 실행 결과를 모델에게 돌려줄 때 사용하는 메시지 역할이다. `tool_call_id`로 호출 요청과 실행 결과를 연결한다.

### 전체 처리 순서

1. `tools`와 사용자의 날씨 질문을 모델에게 보낸다.
2. 모델이 `tool_calls`로 `get_current_weather`와 `city`, `units` 인자를 반환한다.
3. Python 코드가 함수 이름과 인자를 검사한 뒤 OpenWeatherMap API를 호출한다.
4. 날씨 결과를 `role="tool"` 메시지로 `messages`에 추가한다.
5. 누적된 `messages`를 모델에게 다시 보내 최종 자연어 답변을 받는다.

첫 번째 모델 요청은 **사용할 함수와 인자를 결정하는 단계**이다. 두 번째 모델 요청은 **함수 실행 결과를 사용자용 답변으로 바꾸는 단계**이다.

`gpt-5.6-luna`를 Chat Completions의 함수 도구와 함께 사용할 때는 `reasoning_effort="none"`을 명시한다. OpenAI Platform의 Chat 화면에서는 `Model → Reasoning effort → none`으로 변경한 뒤 `Local tools → Function`을 켠다. 웹 Chat은 모델이 선택한 함수 이름과 인자를 확인하지만, PyCharm의 Python 함수나 OpenWeatherMap API를 대신 실행하지는 않는다.


### strict 도구 스키마와 첫 번째 모델 요청

도구 정의는 모델이 선택할 수 있는 함수 이름과 인자 형태를 제한한다.
- `city`와 `units`를 모두 required로 두고 추가 속성을 막은 뒤, 첫 응답을 다음 셀의 `tool_calls` 추출에 사용한다.
- `reasoning_effort="none"`은 `gpt-5.6-luna`와 Chat Completions 함수 도구를 함께 사용하기 위한 호환성 설정이다.


In [7]:
# tools == Chat Completions 모델이 선택할 수 있는 도구 목록
tools = [
    {
        "type": "function", # 도구의 타입이 함수임을 지정
        "function": { # 지정되는 함수에 대한 설명

            "name":"get_current_weather", # LLM 모델이 반환할 함수 이름

            "description":"주어진 도시에 대한 현재 날씨를 반환한다.", # 모델이 이 도구를 선택할 상황을 판단할 수 있도록 하는 설명

            "strict": True, # arguments가 아래 설정을 안정적으로 따르게 강제함

            "parameters": {  # 모델이 만들 함수 인자의 구조
                "type": "object",  # city와 units를 {이름: 값} 형태로 묶어서 받음
                "properties": { # object 안에 들어갈 인자 정의
                    "city": {
                        "type": "string",
                        "description": "OpenWeatherMap에서 조회할 영문 도시 이름(예: Seoul, Busan)", # 모델이 city 값을 선택할 때 참고
                    },

                    "units": {
                        "type": "string",
                        "enum": ["metric", "imperial"], # 허용 값을 섭씨·화씨 단위로 제한
                        "description": "metric은 섭씨, imperial은 화씨 단위이다.",
                    },
                },
                "required": ["city", "units"], # strict 모드에서 두 인자를 모두 필수로 지정
                "additionalProperties": False, # city·units 이외의 인자 생성 차단
            }
        }
    }
]


# messages는 모델이 함수 선택을 판단할 대화 이력
messages = [
    {"role": "system", "content": "너는 도구 결과를 근거로 답하는 챗봇이다."}, # 답변 원칙
    {"role": "user", "content": "오늘 서울 날씨를 알려 줘."}, # 실제 날씨 조회 요청
]

response = client.chat.completions.create(
    model='gpt-5.6-luna',
    messages=messages,
    tools=tools,  # 모델이 선택 가능한 함수, 함수 호출 규칙 전달
    reasoning_effort="none", # Chat Completions 함수 도구 호환 설정
)

### 모델이 반환한 `tool_calls` 확인

첫 번째 요청에서 반환된 `assistant` 메시지를 꺼낸다. 모델이 날씨 함수가 필요하다고 판단했다면 `tool_calls`에 호출 정보가 들어 있다.

각 `tool_call`은 호출을 구분하는 `id`, 함수 이름인 `function.name`, JSON 문자열 인자인 `function.arguments`를 가진다. 이 시점은 함수 실행 전 단계이다.


In [8]:
# 모델의 대답(assistant) 얻어오기
# -> 모델이 등록한 도구 중 '날씨함수'가 필요하다고 판단하면
#    해당 메시지 중 'tool_calls' 부분에 호출 정보를 넣어 둔다.

response_message = response.choices[0].message

tool_calls = response_message.tool_calls or []

print(tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_d2IUJo3t4o0VDHfUKoo3HeDN', function=Function(arguments='{"city":"Seoul","units":"metric"}', name='get_current_weather'), type='function')]


### 허용된 함수를 실행하고 결과 추가

`tool_calls`에는 모델이 만든 호출 요청만 있다. Python 코드는 함수 이름을 `available_functions`에서 확인한 뒤 실제 함수를 실행한다.

`function.arguments`는 JSON 문자열이므로 `json.loads()`로 Python 딕셔너리로 변환한다. 함수 실행 결과는 `role="tool"`과 `tool_call_id`를 포함한 메시지로 추가한다.


In [9]:
available_functions = {"get_current_weather": get_current_weather} # 허용할 함수 이름과 실제 함수 연결

messages.append(response_message) # tool_calls가 들어 있는 assistant 메시지 보존

# 한 응답에 여러 tool_call이 들어 올 수 있으므로 목록을 순회
for tool_call in tool_calls:
    function_name = tool_call.function.name # 모델이 반환한 문자열 함수 이름

    # 허용 목록에 없는 함수는 실행하지 않음
    if function_name not in available_functions:
        raise ValueError(f"허용하지 않은 함수 호출이다: {function_name}")

    function_args = json.loads(tool_call.function.arguments) # JSON 문자열 → Python 딕셔너리
    function_response = available_functions[function_name](**function_args) # 딕셔너리를 키워드 인자로 펼쳐 함수 실행

    # 함수 실행 결과를 원래 호출 ID와 연결해 tool 메시지로 추가
    messages.append(
        {
            "role": "tool", # 함수 실행 결과 메시지
            "tool_call_id": tool_call.id, # 요청과 결과를 연결하는 ID
            "content": function_response, # 날씨 정보 JSON 문자열
        }
    )

### 함수 결과로 최종 답변 생성

`messages`에는 `system → user → assistant(tool_calls) → tool(날씨 결과)` 순서의 대화 이력이 들어 있다. 이 이력을 모델에게 다시 보내면 함수 결과를 사용자가 읽을 수 있는 답변으로 정리한다.

따라서 Chat Completions의 도구 호출은 첫 번째 요청으로 끝나지 않는다. 함수 실행 결과를 포함한 두 번째 요청이 필요하다.


In [11]:
response = client.chat.completions.create(
    model = 'gpt-5.6-luna',
    messages = messages, # 이전 메시지 + 함수 실행 결과가 담겨 있는 메시지
    tools = tools,
    reasoning_effort="none",
)

final_answer = response.choices[0].message.content
print(final_answer)

오늘 서울은 **구름이 조금 끼고**, 현재 기온은 **약 37.8°C**, 습도는 **41%**입니다.


### 전체 도구 호출 과정을 함수로 묶기

`run_conversation(prompt)`은 사용자 질문을 입력받아 첫 번째 모델 요청, 함수 실행, 두 번째 모델 요청을 순서대로 처리한다. 앞에서 나누어 실행한 과정을 하나의 함수로 묶는 단계이다.

모델이 도구를 선택하지 않으면 첫 번째 응답의 텍스트를 바로 반환한다. 도구를 선택하면 허용된 함수를 실행하고, 그 결과를 모델에게 다시 전달해 최종 답변을 반환한다.


In [19]:
# 앞선 과정을 하나로 묶어서 prompt만 전달 받으면 모델이 함수 결과를 이용해서 답을 해주는 동작 구현
def run_conversation(prompt):
    message = [{
        'role' : 'system',
        'content' : '너는 도구 결과를 근거로 답하는 챗봇이다.'
    },
        {
            'role' : 'user',
            'content' : prompt
        }]

    # 1. 사용자 -> 모델 첫 번째 요청
    # 도구 사용 여부와 함수 전달 인자 결정
    response = client.chat.completions.create(
        model='gpt-5.6-luna',
        messages=message,
        tools=tools,
        tool_choice = 'auto',
        reasoning_effort="none"
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls or []

    # 2-1. 도구 (get_current_weather 함수)가 필요 없다고 [] 반환
    if not tool_calls:
        return tool_calls
    # 2-2. 도구 (get_current_weather 함수)가 필요하다고 응답 반환
    message.append(response_message)

    # 3. 도구가 필요하다고 하고, 등록된 도구 목록 (available_functions)에 존재하면
    #    함수 실행 후 결과를 message에 담기
    available_functions = {
        'get_current_weather': get_current_weather
    }

    for tool_call in tool_calls:
        function_name = tool_call.function.name # 모델이 반환한 문자열 함수 이름

        # 허용 목록에 없는 함수는 실행하지 않음
        if function_name not in available_functions:
            raise ValueError(f"허용하지 않은 함수 호출이다: {function_name}")

        function_args = json.loads(tool_call.function.arguments) # JSON 문자열 → Python 딕셔너리
        function_response = available_functions[function_name](**function_args) # 딕셔너리를 키워드 인자로 펼쳐 함수 실행

        # 함수 실행 결과를 원래 호출 ID와 연결해 tool 메시지로 추가
        messages.append(
            {
                "role": "tool", # 함수 실행 결과 메시지
                "tool_call_id": tool_call.id, # 요청과 결과를 연결하는 ID
                "content": function_response, # 날씨 정보 JSON 문자열
            }
        )

    # 4. 사용자 -> 모델 두 번째 요청
    #   - 함수 결과를 이용해서 사용자에게 전달할 답변을 반환
    final_response = client.chat.completions.create(
        model='gpt-5.6-luna',
        messages=messages,
        tools = tools,
        reasoning_effort="none"
    )

    return final_response.choices[0].message.content

#. 5. 답변 출력

print(run_conversation("서울의 현재 날씨를 알려줘."))

BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_ff70AJZAY5AFcAMLNwL4L7Fg", 'type': 'invalid_request_error', 'param': 'messages.[5].role', 'code': None}}

## Responses API 함수 호출 방식

Responses API도 `함수 호출 요청 → Python 함수 실행 → 함수 결과 전달 → 최종 답변` 순서를 사용한다. 신규 구현에서는 추론, 도구 호출, 멀티턴 대화를 하나의 API에서 다루기 쉬운 Responses API를 우선한다.

Chat Completions와 비교하면 사용하는 이름과 구조가 달라진다.

- 함수 스키마를 `function` 안에 중첩하지 않고 `name`, `description`, `parameters`를 바로 작성한다.
- 모델의 호출 요청은 `response.output` 안의 `function_call` 항목으로 반환된다.
- 함수 실행 결과는 `function_call_output` 항목으로 전달한다.
- 최종 텍스트는 `response.output_text`로 꺼낼 수 있다.

아래 첫 번째 요청에서는 `tool_choice`로 날씨 함수를 지정해 `function_call`을 확실하게 확인한다. 실제 서비스에서는 모델이 도구 사용 여부를 판단하도록 자동 선택을 사용할 수 있다.

- [GPT-5.6 사용 가이드](https://developers.openai.com/api/docs/guides/latest-model)
- [Function Calling 가이드](https://developers.openai.com/api/docs/guides/function-calling)
- [Responses 생성 API](https://developers.openai.com/api/reference/resources/responses/methods/create)


In [21]:
responses_tools = [
    {
        "type": "function", # 함수 도구 정의
        "name": "get_current_weather", # 모델이 반환할 함수 이름
        "description": "주어진 도시에 대한 현재 날씨를 반환한다.", # 함수 선택 기준
        "strict": True, # 인자가 아래 JSON Schema를 따르도록 제한
        "parameters": { # 모델이 만들 함수 인자의 구조
            "type": "object", # city와 units를 {이름: 값} 형태로 묶음
            "properties": { # object 안에 들어갈 인자 정의
                "city": {
                    "type": "string",
                    "description": "OpenWeatherMap에서 조회할 영문 도시 이름(예: Seoul, Busan)",
                },
                "units": {"type": "string", "enum": ["metric", "imperial"]}, # 허용 단위 제한
            },
            "required": ["city", "units"], # 두 인자를 모두 필수로 지정
            "additionalProperties": False, # 정의하지 않은 인자 생성 차단
        },
    }
]

# input_items에 사용자 입력부터 함수 결과까지 순서대로 누적
input_items = [
    {"role": "user", "content": "오늘 서울 날씨를 알려 줘."}, # 첫 사용자 질문
]

# 첫 번째 요청: 날씨 함수의 function_call 생성
response = client.responses.create(
    model="gpt-5.6-luna", # 함수와 인자를 선택할 모델
    input=input_items, # 사용자 질문 전달
    tools=responses_tools, # 사용할 수 있는 함수 정의 전달
    # tool_choice={"type": "function", "name": "get_current_weather"}, # 수업에서는 함수 호출을 강제
)

input_items += response.output # 첫 응답의 모든 항목을 대화 상태에 보존

# response.output에서 실제 함수 호출 항목만 선택
function_calls = [item for item in response.output if item.type == "function_call"]

# 함수 호출을 강제했는데 결과가 없으면 실행 중단
if not function_calls:
    raise RuntimeError("함수 호출이 반환되지 않았다. 첫 응답의 output을 확인한다.")

# 반환된 function_call을 하나씩 검증하고 실행
for item in function_calls:
    # 허용 목록에 없는 함수는 실행하지 않음
    if item.name not in available_functions:
        raise ValueError(f"허용하지 않은 함수 호출이다: {item.name}")

    function_args = json.loads(item.arguments) # JSON 문자열 → Python 딕셔너리
    function_response = available_functions[item.name](**function_args) # 허용된 함수 실행

    # Responses API는 함수 결과를 function_call_output 항목으로 추가
    input_items.append(
        {
            "type": "function_call_output", # 함수 실행 결과 항목
            "call_id": item.call_id, # 호출 요청과 결과 연결
            "output": function_response, # 날씨 정보 JSON 문자열
        }
    )

# 두 번째 요청: 함수 결과를 최종 자연어 답변으로 변환
final_response = client.responses.create(
    model="gpt-5.6-luna", # 함수 결과를 설명할 모델
    input=input_items, # 사용자 입력·함수 호출·함수 결과 전달
    tools=responses_tools, # 같은 함수 정의 전달
)

print(final_response.output_text) # 최종 텍스트만 출력

오늘 서울은 **구름이 조금 끼고**, 현재 기온은 **약 36.3°C**, 습도는 **45%**입니다.  
더운 날씨이니 수분을 충분히 섭취하고 야외 활동 시 주의하세요.
